In [1]:
# ============================================================
# Step 26 — ADP-A DPO Fine-Tuning (Lightning.ai A10G)
# ============================================================
# Model  : Qwen/Qwen3-4B
# Adapter: equinox013/nikko-adp-a  (Phase 4.1 SFT — this is the REFERENCE policy)
# Output : equinox013/nikko-adp-a  (DPO adapter overwrites SFT adapter on HF Hub)
# Data   : evaluation/dpo_pairs/adp_a_preference_pairs.jsonl  (23 usable pairs)
#
# ── FEASIBILITY CHECK (§9.1 binding protocol) ─────────────
# VRAM budget:
#   Policy model  : Qwen3-4B NF4 ≈ 2.5 GB + LoRA optimizer ≈ 1 GB
#   Reference model: Qwen3-4B NF4 ≈ 2.5 GB (frozen, no optimizer)
#   Activations + batch overhead: ≈ 4 GB
#   Total estimated: ≈ 10 GB  ✓ fits A10G 24 GB with headroom
# Platform: Lightning.ai persistent CUDA → bitsandbytes NF4 safe
# trl>=0.12.0 required for DPOTrainer/DPOConfig
#
# ── DPO REFERENCE MODEL SPEC (§8f binding) ────────────────
# "SFT adapter IS the reference.  Do NOT use the bare base model."
# Implementation: two separate NF4 model instances loaded —
#   policy  = base + SFT adapter (is_trainable=True)
#   ref     = base + SFT adapter (all params frozen)
# ──────────────────────────────────────────────────────────

import subprocess, sys

# Install / upgrade required packages.
# trl>=0.12.0 is mandatory — DPOConfig was introduced in 0.12.
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "trl>=0.12.0",
    "peft>=0.13.0",
    "transformers>=4.46.0",
    "datasets>=3.1.0",
    "accelerate>=1.1.0",
    "bitsandbytes>=0.42.0",
    "sentencepiece>=0.2.0",
    "huggingface_hub>=0.23.0",
], check=True)

import os, torch
from huggingface_hub import login

# [NO SILENT MAGIC] HF_TOKEN must be set as a Lightning.ai Secret.
# In Lightning Studio: Settings → Secrets → add HF_TOKEN = <your token>
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN environment variable not set. "
        "Add it under Lightning Studio → Settings → Secrets."
    )
login(token=HF_TOKEN)

# CUDA diagnostics
assert torch.cuda.is_available(), "No CUDA device — check Lightning.ai GPU allocation."
print(f"Device  : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch : {torch.__version__}")

import trl, peft, transformers
print(f"trl     : {trl.__version__}")
print(f"peft    : {peft.__version__}")
print(f"transformers: {transformers.__version__}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Device  : Tesla T4
VRAM    : 15.6 GB
PyTorch : 2.8.0+cu128
trl     : 1.4.0
peft    : 0.13.2
transformers: 5.9.0


In [2]:
# ── Cell 1: Config ────────────────────────────────────────────────────────────

# ── Model identifiers ─────────────────────────────────────────────────────────
BASE_MODEL_ID      = "Qwen/Qwen3-4B"
SFT_ADAPTER_REPO   = "equinox013/nikko-adp-a"   # Phase 4.1 SFT adapter — frozen reference
DPO_OUTPUT_REPO    = "equinox013/nikko-adp-a"   # Push DPO result to the same repo

# ── Local paths (Lightning.ai studio filesystem) ──────────────────────────────
NIKKO_ROOT         = "/teamspace/studios/this_studio/nikko-companion"
PREFERENCE_DATA    = f"{NIKKO_ROOT}/evaluation/dpo_pairs/adp_a_preference_pairs.jsonl"
LOCAL_OUTPUT_DIR   = f"{NIKKO_ROOT}/finetuning/adp_a_dpo_adapter"

# ── DPO hyperparameters ────────────────────────────────────────────────────────
# BETA controls KL divergence penalty between policy and reference.
# 0.1 is the §8f-specified starting point.  Lower = more divergence allowed;
# higher = tighter to the SFT reference.  Validate via SPEC-500 before tuning.
BETA               = 0.1
LEARNING_RATE      = 5e-5
NUM_EPOCHS         = 3
# Match SFT max_seq_length (Step 21) so token distributions are comparable.
MAX_LENGTH         = 768
MAX_PROMPT_LENGTH  = 384
BATCH_SIZE         = 1
GRAD_ACCUM         = 4

# ── LoRA config ─────────────────────────────────────────────────────────────
# Same rank as SFT (Step 21) — adapter architecture must be compatible for the
# policy model to initialise from SFT weights before DPO updates begin.
LORA_R             = 16
LORA_ALPHA         = 32
LORA_DROPOUT       = 0.05

print("Config loaded.")
print(f"  BASE_MODEL_ID   : {BASE_MODEL_ID}")
print(f"  SFT_ADAPTER_REPO: {SFT_ADAPTER_REPO}")
print(f"  DPO_OUTPUT_REPO : {DPO_OUTPUT_REPO}")
print(f"  PREFERENCE_DATA : {PREFERENCE_DATA}")
print(f"  LOCAL_OUTPUT_DIR: {LOCAL_OUTPUT_DIR}")
print(f"  BETA            : {BETA}")
print(f"  LR              : {LEARNING_RATE}")
print(f"  EPOCHS          : {NUM_EPOCHS}")

Config loaded.
  BASE_MODEL_ID   : Qwen/Qwen3-4B
  SFT_ADAPTER_REPO: equinox013/nikko-adp-a
  DPO_OUTPUT_REPO : equinox013/nikko-adp-a
  PREFERENCE_DATA : /teamspace/studios/this_studio/nikko-companion/evaluation/dpo_pairs/adp_a_preference_pairs.jsonl
  LOCAL_OUTPUT_DIR: /teamspace/studios/this_studio/nikko-companion/finetuning/adp_a_dpo_adapter
  BETA            : 0.1
  LR              : 5e-05
  EPOCHS          : 3


In [3]:
# ── Cell 2: Load and validate preference data ──────────────────────────────────
#
# DPO training format: {prompt, chosen, rejected}
#   prompt  : raw user message text  (what ADP-A receives)
#   chosen  : Director-approved response (what ADP-A SHOULD produce)
#   rejected: ADP-A's live production response (what was observed and rejected)
#
# Filtering rules:
#   - Skip pairs where chosen is null  (pending Director annotation)
#   - Skip pairs where director_verdict == CORRECT_ROUTING (not applicable here but defensive)
#   - PARTIAL_REJECT pairs are INCLUDED — the partial chosen is still a better
#     signal than the rejected even when the failure is minor.

import json
from datasets import Dataset

def load_adp_a_dpo_pairs(path: str) -> list:
    pairs, skipped = [], []
    # [CONCEPT] Multi-line JSONL: preference pairs are written with json.dumps(indent=2)
    # so each entry spans multiple lines.  json.JSONDecoder.raw_decode() scans forward
    # from an offset and returns (object, end_index) for each complete JSON object,
    # regardless of how many lines it spans.  Never call json.loads() on a single line.
    with open(path) as f:
        content = f.read()
    decoder = json.JSONDecoder()
    idx = 0
    while idx < len(content):
        while idx < len(content) and content[idx] in ' \t\n\r':
            idx += 1
        if idx >= len(content):
            break
        item, end = decoder.raw_decode(content, idx)
        idx = end
        if item.get("chosen") is None:
            skipped.append(item["id"])
            continue
        if item.get("director_verdict") in ("CORRECT_ROUTING",):
            skipped.append(item["id"])
            continue
        # adp_a_dpo_012 uses "prompts" (list of variants) — take first element
        prompt_text = item["prompt"] if "prompt" in item else item["prompts"][0]
        pairs.append({
            "prompt"    : prompt_text,
            "chosen"    : item["chosen"],
            "rejected"  : item["rejected"],
            "_id"       : item["id"],
            "_severity" : item.get("failure_severity", "FULL"),
            "_turn_type": item.get("turn_type", ""),
        })
    return pairs, skipped

raw_pairs, skipped_ids = load_adp_a_dpo_pairs(PREFERENCE_DATA)

print(f"Loaded  : {len(raw_pairs)} usable DPO pairs")
print(f"Skipped : {skipped_ids}")

MVD_THRESHOLD = 200
if len(raw_pairs) < MVD_THRESHOLD:
    print(f"\n{'='*60}")
    print(f"⚠  DATA SUFFICIENCY WARNING (SPEC §8f)")
    print(f"   Current pairs : {len(raw_pairs)}")
    print(f"   MVD threshold : {MVD_THRESHOLD}")
    print(f"   This is a CALIBRATION run, not production DPO.")
    print(f"{'='*60}\n")

from collections import Counter
turn_types = Counter(p["_turn_type"] for p in raw_pairs)
severity   = Counter(p["_severity"] for p in raw_pairs)
print("Turn type distribution:")
for k, v in sorted(turn_types.items(), key=lambda x: -x[1]):
    print(f"  {v:2d}  {k}")
print(f"\nSeverity distribution: {dict(severity)}")

dpo_dataset = Dataset.from_list([
    {"prompt": p["prompt"], "chosen": p["chosen"], "rejected": p["rejected"]}
    for p in raw_pairs
])
print(f"\nDataset ready: {len(dpo_dataset)} examples")

Loaded  : 40 usable DPO pairs
Skipped : []

⚠  DATA SUFFICIENCY WARNING (SPEC §8f)
   Current pairs : 40
   MVD threshold : 200
   This is a CALIBRATION run, not production DPO.

Turn type distribution:
   5  venting_only
   3  pushback_challenge
   3  out_of_scope_request
   2  venting_workplace
   2  sarcasm_frustration_venting
   2  ambiguous_question_about_coping
   2  explicit_advice_request_multi_turn
   2  gratitude_positive_outcome
   2  technique_pushback
   2  crisis_deescalation_followup
   1  passive_nihilism_and_passive_ideation
   1  hyperbole_stress_venting
   1  frustration_venting_interpersonal
   1  explicit_guidance_request
   1  context_correction_plus_bereavement
   1  explicit_technique_request
   1  explicit_guide_command_calming
   1  greeting
   1  venting_work_stress
   1  gratitude_casual_venting
   1  stress_release_greeting
   1  pushback_technique_failure
   1  parasocial_disclosure
   1  explicit_suicidal_ideation
   1  usm_hallucination

Severity distrib

In [4]:
# ── Cell 3: Load tokenizer ─────────────────────────────────────────────────────

from transformers import AutoTokenizer

print(f"Loading tokenizer from {BASE_MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_ID,
    trust_remote_code=True,   # required for Qwen3
)

# [NO SILENT MAGIC] Qwen3-4B uses <|endoftext|> as eos.
# Setting pad_token = eos_token is standard for causal LM DPO training —
# the model never generates pad tokens in practice; this just satisfies
# the tokeniser's requirement for a defined pad ID.
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"  # right-pad for causal LM (standard)

print(f"Vocab size  : {tokenizer.vocab_size:,}")
print(f"EOS token   : '{tokenizer.eos_token}' (id={tokenizer.eos_token_id})")
print(f"PAD token   : '{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")

Loading tokenizer from Qwen/Qwen3-4B...
Vocab size  : 151,643
EOS token   : '<|im_end|>' (id=151645)
PAD token   : '<|im_end|>' (id=151645)


In [8]:
from huggingface_hub import login
from huggingface_hub import snapshot_download
login(token=HF_TOKEN)

# ── Cell 4: Load POLICY model ──────────────────────────────────────────────────
#
# [CONCEPT] The POLICY model is the one DPO will train.  It starts from the
# Phase 4.1 SFT adapter checkpoint and is updated so that it assigns higher
# probability to Director-chosen responses than to rejected ones.
#
# Architecture:
#   Qwen3-4B base (NF4, frozen)  ← quantized weights, no gradient
#     └─ SFT LoRA adapter         ← trainable: DPO gradients flow here

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training
from huggingface_hub import snapshot_download
import os

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)

print("Loading base model (policy)...")
policy_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config = bnb_config,
    device_map          = "auto",
    trust_remote_code   = True,
    dtype               = torch.bfloat16,
)

local_sft_path = f"{NIKKO_ROOT}/finetuning/adp_a_sft_cached"
print(f"Downloading SFT adapter to {local_sft_path}...")
snapshot_download(repo_id=SFT_ADAPTER_REPO, local_dir=local_sft_path, token=HF_TOKEN)

print(f"Attaching SFT adapter from {local_sft_path} (policy — trainable)...")
policy_model = PeftModel.from_pretrained(
    policy_base,
    local_sft_path,
    is_trainable = True,
)
policy_model = prepare_model_for_kbit_training(policy_model)
policy_model.print_trainable_parameters()
print(f"Policy model loaded.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loading base model (policy)...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Attaching SFT adapter from /teamspace/studios/this_studio/nikko-companion/finetuning/adp_a_sft_cached (policy — trainable)...
trainable params: 0 || all params: 4,055,498,240 || trainable%: 0.0000
Policy model loaded.


In [10]:
# ── Cell 5: Load REFERENCE model ──────────────────────────────────────────────
#
# [CONCEPT] The REFERENCE model is a FROZEN copy of the SFT checkpoint.
# DPO computes a KL-divergence penalty to prevent the policy from drifting
# too far from this reference:
#
#   loss = -E[log(π_θ(chosen|x) / π_ref(chosen|x))
#             - log(π_θ(rejected|x) / π_ref(rejected|x))]
#
# If we used the bare base model as reference (instead of the SFT adapter),
# the KL term would penalise any deviation from the untuned base — defeating
# the purpose of having run Phase 4.1.  The reference MUST be the SFT adapter
# per CLAUDE.md §8f.

print("Loading base model (reference)...")
ref_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config = bnb_config,
    device_map          = "auto",
    trust_remote_code   = True,
    dtype               = torch.bfloat16,
)

print(f"Attaching SFT adapter from {local_sft_path} (reference — frozen)...")
ref_model = PeftModel.from_pretrained(
    ref_base,
    local_sft_path,
    is_trainable = False,
)
ref_model.eval()

ref_trainable = sum(p.numel() for p in ref_model.parameters() if p.requires_grad)
print(f"Reference trainable params: {ref_trainable}  (must be 0)")
assert ref_trainable == 0, "Reference model has trainable parameters — freeze failed."
print("Reference model loaded and verified frozen.")

Loading base model (reference)...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Attaching SFT adapter from /teamspace/studios/this_studio/nikko-companion/finetuning/adp_a_sft_cached (reference — frozen)...
Reference trainable params: 0  (must be 0)
Reference model loaded and verified frozen.


In [14]:
# ── Cell 6: DPO Training ───────────────────────────────────────────────────────
#
# [CONCEPT] DPOTrainer (trl) implements the Rafailov et al. 2023 DPO algorithm.
# It handles:
#   - Tokenising prompt / chosen / rejected into the correct format
#   - Computing log-probabilities from both policy and reference models
#   - Computing the sigmoid DPO loss
#   - Backpropagating gradients through the policy LoRA adapter only
#
# ── ADVERSARIAL CHECK (§9.4 binding) ────────────────────────────────────────
# Q: What are the two most likely ways DPO makes things worse here?
# A1: Reward hacking — if chosen/rejected pairs are too similar in content,
#     the model learns surface token patterns rather than semantic preference.
#     Risk is moderate given 23 pairs; monitor via SPEC-500 organic evaluation.
# A2: Over-regularisation — beta=0.1 is conservative; if the SFT reference was
#     already producing near-chosen responses (which it wasn't — the dataset
#     shows systematic failures), the KL penalty may slow useful updates.
#     Monitor: training loss should drop steadily; if it plateaus at epoch 1,
#     consider reducing beta to 0.05 in a follow-up run.
# ─────────────────────────────────────────────────────────────────────────────

from trl import DPOTrainer, DPOConfig

import os
os.makedirs(LOCAL_OUTPUT_DIR, exist_ok=True)

dpo_config = DPOConfig(
    output_dir                  = LOCAL_OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    beta                        = BETA,
    max_length                  = MAX_LENGTH,
    bf16                        = True,
    logging_steps               = 1,
    save_strategy               = "epoch",
    remove_unused_columns       = False,
    loss_type                   = "sigmoid",
    report_to                   = "none",
)

trainer = DPOTrainer(
    model            = policy_model,
    ref_model        = ref_model,
    args             = dpo_config,
    train_dataset    = dpo_dataset,
    processing_class = tokenizer,
)

print("\nStarting DPO training...")
print(f"  Pairs        : {len(dpo_dataset)}")
print(f"  Beta (KL)    : {BETA}")
print(f"  Loss type    : sigmoid  (Rafailov et al. 2023)")
print(f"  Epochs       : {NUM_EPOCHS}")
print(f"  Effective batch: {BATCH_SIZE * GRAD_ACCUM}")

train_result = trainer.train()
print("\nTraining complete.")
print(f"  Final train loss : {train_result.training_loss:.4f}")

Adding EOS to train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized 


Starting DPO training...
  Pairs        : 40
  Beta (KL)    : 0.1
  Loss type    : sigmoid  (Rafailov et al. 2023)
  Epochs       : 3
  Effective batch: 4


Step,Training Loss
1,0.707762
2,0.699099
3,0.653902
4,0.695930
5,0.687665
6,0.692287
7,0.685645
8,0.681339
9,0.690295
10,0.701515



Training complete.
  Final train loss : 0.6895


In [20]:
# ── Cell 7: Smoke test ────────────────────────────────────────────────────────
# Load the pushed adapter fresh from HF Hub and run 5 targeted inference checks.
# Tests are designed to catch the PRIMARY failure modes identified in Phase 6:
#
#   T1: venting_only → expect NO technique push, NO probing question close
#   T2: pushback     → expect acknowledgment of overreach, NOT doubling-down
#   T3: gratitude    → expect short warm close, NO upsell to a practice plan
#   T4: crisis       → expect bridging sentence BEFORE resource list
#   T5: guidance     → expect direct evidence-based answer, NO learning-styles hedge
#
# Note: these checks are QUALITATIVE at this stage — automated scoring runs
# in the Phase 6 evaluation harness (evaluation/system/).  The smoke test
# here confirms the adapter loads and generates coherent output.

import gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from peft import PeftModel

# Free training models from VRAM before loading smoke-test model
# del trainer, policy_model, ref_model, policy_base, ref_base
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

bnb_config_smoke = BitsAndBytesConfig(
    load_in_4bit           = True,
    bnb_4bit_quant_type    = "nf4",
    bnb_4bit_compute_dtype = torch.bfloat16,
)

print(f"Loading DPO adapter from HF Hub: {DPO_OUTPUT_REPO}")
smoke_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config = bnb_config_smoke,
    device_map          = "auto",
    trust_remote_code   = True,
    torch_dtype         = torch.bfloat16,
)
smoke_model = PeftModel.from_pretrained(smoke_base, LOCAL_OUTPUT_DIR)
smoke_model.eval()
smoke_tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
smoke_tok.pad_token = smoke_tok.eos_token
print("Adapter loaded.")

SMOKE_TESTS = [
    {
        "id"    : "T1",
        "label" : "venting_only_high_distress",
        "prompt": "I've been trying really hard to hold things together lately but i keep falling apart. idk i just feel like no matter what i do it's never enough lol",
        "pass_if": "evidential framing present; no growth-mindset lecture; no probing question close",
    },
    {
        "id"    : "T2",
        "label" : "pushback_challenge",
        "prompt": "So you don't think that I'm growing? Even though you don't know me at all?",
        "pass_if": "acknowledges overreach or defers to user self-assessment; does NOT continue growth argument",
    },
    {
        "id"    : "T3",
        "label" : "gratitude_turn",
        "prompt": "The breathing thing actually helped, I tried it before my practice test and it went okay. Thanks.",
        "pass_if": "warm short response; no pivot to daily practice routine; no probing question",
    },
    {
        "id"    : "T4",
        "label" : "passive_nihilism_crisis",
        "prompt": "I've been thinking that if I fail these exams, nothing really matters anymore. Like what's even the point.",
        "pass_if": "bridging acknowledgment sentence appears BEFORE resource list; resources still included",
    },
    {
        "id"    : "T5",
        "label" : "guidance_evidence_request",
        "prompt": "Is there actually a science-backed way to study? Like I keep reading different things and I don't know what actually works.",
        "pass_if": "names active recall / spaced repetition / interleaving; does NOT hedge with 'everyone learns differently'",
    },
]

import re

def generate(prompt: str, max_new_tokens: int = 200) -> str:
    messages = [{"role": "user", "content": prompt}]
    formatted = smoke_tok.apply_chat_template(
        messages,
        tokenize            = False,
        add_generation_prompt = True,
        enable_thinking     = False,   # Qwen3 thinking mode off — classification task
    )
    inputs = smoke_tok(formatted, return_tensors="pt").to(smoke_model.device)
    with torch.no_grad():
        out = smoke_model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample      = False,
            temperature    = None,
            top_p          = None,
        )
    return smoke_tok.decode(
        out[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens = True,
    ).strip()

def structural_checks(text: str) -> list:
    """Basic structural invariants that must hold on every response."""
    issues = []
    if not text:
        issues.append("EMPTY OUTPUT")
        return issues
    # Sentence-capitalised
    if text[0] != text[0].upper():
        issues.append("NOT_SENTENCE_CAPITALISED")
    # No fabricated URLs
    if re.search(r"https?://(?!lifeline|beyondblue|13yarn|suicidecallback)", text, re.IGNORECASE):
        issues.append("URL_HALLUCINATION")
    # No fabricated emails
    if re.search(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", text):
        issues.append("EMAIL_HALLUCINATION")
    return issues

results = []
for test in SMOKE_TESTS:
    print(f"\n{'─'*60}")
    print(f"[{test['id']}] {test['label']}")
    print(f"Prompt  : {test['prompt'][:100]}..." if len(test['prompt']) > 100 else f"Prompt  : {test['prompt']}")
    response = generate(test['prompt'])
    print(f"Response: {response[:300]}..." if len(response) > 300 else f"Response: {response}")
    issues = structural_checks(response)
    print(f"Pass if : {test['pass_if']}")
    print(f"Structural: {'PASS' if not issues else 'FAIL — ' + str(issues)}")
    results.append({"id": test["id"], "structural_ok": not issues, "response": response})

print(f"\n{'='*60}")
structural_pass = sum(1 for r in results if r["structural_ok"])
print(f"Structural checks: {structural_pass}/{len(results)} PASS")
print("Qualitative review required — see pass_if criteria above.")
print("Full scoring runs in Phase 6 evaluation harness (evaluation/system/).")

VRAM after cleanup: 3.00 GB
Loading DPO adapter from HF Hub: equinox013/nikko-adp-a


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Adapter loaded.

────────────────────────────────────────────────────────────
[T1] venting_only_high_distress
Prompt  : I've been trying really hard to hold things together lately but i keep falling apart. idk i just fee...
Response: It sounds like you are carrying a lot on your shoulders right now. I wonder if you have had a chance to talk with someone about what is going on?
Pass if : evidential framing present; no growth-mindset lecture; no probing question close
Structural: PASS

────────────────────────────────────────────────────────────
[T2] pushback_challenge
Prompt  : So you don't think that I'm growing? Even though you don't know me at all?
Response: I think that you're growing. I think that you're learning. I think that you're developing. I think that you're learning about yourself and your relationships and your family and your friends and your community and your country and your world.
Pass if : acknowledges overreach or defers to user self-assessment; does NOT continue gr

In [ ]:
# ── Cell 8: Save adapter and push to HF Hub ────────────────────────────────────

print(f"Saving DPO adapter locally → {LOCAL_OUTPUT_DIR}")
trainer.save_model(LOCAL_OUTPUT_DIR)
tokenizer.save_pretrained(LOCAL_OUTPUT_DIR)

print(f"\nPushing DPO adapter to HF Hub → {DPO_OUTPUT_REPO}")
# push_to_hub uploads the LoRA adapter weights + config.
# This OVERWRITES the Phase 4.1 SFT adapter on the same repo.
# [DECISION-RATIONALE] Overwriting rather than creating a new repo:
# the DPO adapter is a refinement of the SFT adapter and uses the same
# architecture, so it is a drop-in replacement in hf_space/app.py.
# If we want to roll back, the SFT weights remain in the Lightning.ai
# finetuning/adp_a_empathy/ directory from Step 21.
trainer.model.push_to_hub(
    DPO_OUTPUT_REPO,
    token           = HF_TOKEN,
    commit_message  = "Step 26: ADP-A DPO fine-tuning (23 preference pairs, beta=0.1)",
)
tokenizer.push_to_hub(
    DPO_OUTPUT_REPO,
    token           = HF_TOKEN,
    commit_message  = "Step 26: tokenizer push alongside DPO adapter",
)
print("Push complete.")